# Hair & Scalp Classifier — Training Notebook

Run this in Google Colab with a **free GPU**: `Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU`.

Steps in this notebook:
1. Confirm GPU is active
2. Get your project code into Colab (clone from GitHub)
3. Install dependencies
4. Download a hair/scalp dataset from Kaggle
5. Reorganize it into the `data/train/<class>/`, `data/val/<class>/` layout `train.py` expects
6. Run training
7. Download the resulting `best_model.pth` and `class_names.json`

See `SETUP_GUIDE.md` in the project root for the full walkthrough, including which Kaggle dataset to use.

In [ ]:
# 1. Confirm you have a GPU (should print info about a Tesla T4 or similar)
!nvidia-smi

## 2. Get your code into Colab

Push your project to a GitHub repo first (see SETUP_GUIDE.md step 3), then clone it here.
Replace the URL below with your own repo URL.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/hair-health-ai.git"  # <-- change this
!git clone $REPO_URL
%cd hair-health-ai

In [ ]:
# 3. Install dependencies (Colab already has torch/torchvision preinstalled
# with GPU support, so we skip reinstalling those two to save time)
!pip install -q grad-cam split-folders kaggle

## 4. Download a dataset from Kaggle

You need a free Kaggle account and an API token:
1. Go to kaggle.com → your profile picture → **Settings** → **API** → **Create New Token**. This downloads a `kaggle.json` file.
2. Run the cell below and upload that `kaggle.json` when prompted.

Two real datasets that fit this project well (pick one, or combine them later):
- `trainingdatapro/bald-men` — Norwood-scale bald/hair-loss staged photos (best match for the 4-stage classifier as designed)
- `abubakar4u900/hair-and-scalp-disease-dataset` — scalp condition photos (dandruff, healthy, etc.) if you'd rather classify conditions instead of/in addition to loss stage

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your kaggle.json here
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
DATASET_SLUG = "trainingdatapro/bald-men"  # <-- change if you pick a different dataset
!kaggle datasets download -d $DATASET_SLUG -p raw_dataset --unzip

## 5. Inspect and reorganize the data

**Important:** Kaggle datasets vary in internal structure. Run the next cell first and actually look at the printed output before continuing — you need to know which folder holds the images and how classes are labeled (folder names, a CSV of labels, or filename patterns) before the reorganization step will work correctly. Adjust the reorganization cell below to match what you see.

In [ ]:
!find raw_dataset -maxdepth 3 | head -50

In [ ]:
# If the dataset is already organized as raw_dataset/<class_name>/*.jpg,
# split-folders can create the train/val split for you in one line:
import splitfolders

INPUT_FOLDER = "raw_dataset"  # <-- point this at the folder containing per-class subfolders
splitfolders.ratio(INPUT_FOLDER, output="data", seed=42, ratio=(0.8, 0.2))

!find data -maxdepth 2

If your chosen dataset instead ships as a flat folder of images plus a CSV of labels (common for some Kaggle sets), you'll need a short custom loop instead of `splitfolders` — copy each image into `raw_dataset_sorted/<label>/` based on its CSV row first, then run the `splitfolders.ratio(...)` cell above pointed at `raw_dataset_sorted`. Ask your AI assistant to write that loop for your specific dataset's CSV format if needed — the column names differ per dataset.

## 6. Train

Start with a small number of epochs to make sure everything runs, then increase once you see it's working.

In [ ]:
!python training/train.py --data_dir data --epochs 15 --lr 0.001 --batch_size 16

## 7. Download your trained model

This downloads two files to your computer: `best_model.pth` and `class_names.json`.
Place both into your local project's `models/` folder (see SETUP_GUIDE.md step 5), then
commit and push — that's what makes the deployed app use your real trained model instead
of demo mode.

In [ ]:
from google.colab import files
files.download("models/best_model.pth")
files.download("models/class_names.json")